# ==============================================================================
# 🏗️ CIVIL & MATERIALS ENGINEERING: CONCRETE COMPRESSIVE STRENGTH VIA DECISION TREE REGRESSOR
# ==============================================================================
### Mathematical Formulation & Theoretical Context:
Concrete compressive strength (measured in Megapascals, MPa) is a highly non-linear function of mixture proportions (cement, slag, fly ash, water, superplasticizer, coarse/fine aggregate, and curing age in days).

A Decision Tree Regressor models this continuous response surface by recursively partitioning feature space into orthogonal hyper-rectangles:
$$\hat{y}(\mathbf{x}) = \sum_{m=1}^M \bar{y}_m \cdot \mathbb{I}(\mathbf{x} \in R_m)$$
Splits are selected to maximize **Variance Reduction**:
$$\Delta \text{Var} = \text{Var}(D) - \left( \frac{N_L}{N}\text{Var}(D_L) + \frac{N_R}{N}\text{Var}(D_R) \right)$$
Where $\text{Var}(D) = \frac{1}{N} \sum_{i=1}^N (y_i - \bar{y})^2$.


In [1]:
# STEP 1: ENVIRONMENT & IMPORTS
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Materials Engineering Analytics Environment ready.")


✅ STEP 1: Materials Engineering Analytics Environment ready.


## 📥 STEP 2 & 3: INGESTION & COLUMN SANITIZATION
Loading the Kaggle Concrete Compressive Strength dataset (1,030 mixture tests).


In [2]:
# STEP 2 & 3: LOAD & SANITIZE
data_path = 'data/concrete/Concrete Compressive Strength.csv'
df = pd.read_csv(data_path)

# Sanitize column names
df.columns = [
    'cement', 'blast_furnace_slag', 'fly_ash', 'water',
    'superplasticizer', 'coarse_aggregate', 'fine_aggregate', 'age', 'compressive_strength'
]

print(f"📊 Dataset Shape: {df.shape[0]:,} formulations, {df.shape[1]} attributes")
print(f"📋 Formulations head:\n{df.head(3)}")


📊 Dataset Shape: 1,030 formulations, 9 attributes
📋 Formulations head:
   cement  blast_furnace_slag  fly_ash  water  superplasticizer  \
0   540.0                 0.0      0.0  162.0               2.5   
1   540.0                 0.0      0.0  162.0               2.5   
2   332.5               142.5      0.0  228.0               0.0   

   coarse_aggregate  fine_aggregate  age  compressive_strength  
0            1040.0           676.0   28             79.986111  
1            1055.0           676.0   28             61.887366  
2             932.0           594.0  270             40.269535  


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features $X$ and target $y$ (`compressive_strength`), checking missing values and physical boundaries.


In [3]:
# STEP 4 & 5: SEGREGATE & HYGIENE
X = df.drop(columns=['compressive_strength'])
y = df['compressive_strength']

print(f"Missing attributes: {X.isnull().sum().sum()}")
print(f"Target Summary (Strength in MPa):")
print(y.describe().round(2))


Missing attributes: 0
Target Summary (Strength in MPa):
count    1030.00
mean       35.82
std        16.71
min         2.33
25%        23.71
50%        34.44
75%        46.14
max        82.60
Name: compressive_strength, dtype: float64


## 🔒 STEP 6: LEAK-FREE TRAIN-TEST SPLIT
80/20 train/test split.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train Mixtures: {X_train.shape[0]}")
print(f"Test Mixtures : {X_test.shape[0]}")


Train Mixtures: 824
Test Mixtures : 206


## ⚙️ STEP 7: SCALE-INVARIANT PREPROCESSOR
Because Decision Trees split strictly on order statistics, **no feature scaling is applied**.
A SimpleImputer is included for production resilience.


In [5]:
# STEP 7: PREPROCESSOR
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])
print("✅ STEP 7: Scale-Invariant Pass-through Pipeline ready.")


✅ STEP 7: Scale-Invariant Pass-through Pipeline ready.


## ⚠️ STEP 8: THE OVERFITTING / ZERO-RESIDUAL TRAP
Demonstrating why unconstrained trees memorize training noise ($R^2=1.0$) but fail in production.


In [6]:
# STEP 8: UNCONSTRAINED VS PRE-PRUNED
unconstrained_tree = DecisionTreeRegressor(max_depth=None, min_samples_split=2, random_state=42)
unconstrained_tree.fit(X_train, y_train)

r2_train_uncon = r2_score(y_train, unconstrained_tree.predict(X_train))
r2_test_uncon = r2_score(y_test, unconstrained_tree.predict(X_test))

print(f"Unconstrained Tree Train R² : {r2_train_uncon*100:.2f}% (Pure Memorization)")
print(f"Unconstrained Tree Test R²  : {r2_test_uncon*100:.2f}% (Variance Collapse)")
print(f"Generalization Drop         : {(r2_train_uncon - r2_test_uncon)*100:.2f}% R² drop!")


Unconstrained Tree Train R² : 99.75% (Pure Memorization)
Unconstrained Tree Test R²  : 80.58% (Variance Collapse)
Generalization Drop         : 19.17% R² drop!


## 🎯 STEP 9: HYPERPARAMETER TUNING VIA GRID SEARCH
Optimizing `max_depth`, `min_samples_leaf`, and `criterion` using 5-Fold Cross Validation.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'tree__max_depth': [4, 6, 8, 10, 12],
    'tree__min_samples_split': [5, 10, 20],
    'tree__min_samples_leaf': [2, 5, 10],
    'tree__criterion': ['squared_error', 'absolute_error', 'poisson']
}

pipe = Pipeline([
    ('prep', preprocessor),
    ('tree', DecisionTreeRegressor(random_state=42))
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(pipe, param_grid=param_grid, cv=cv, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV R²   : {grid.best_score_*100:.2f}%")


🏆 Best Hyperparameters: {'tree__criterion': 'squared_error', 'tree__max_depth': 12, 'tree__min_samples_leaf': 2, 'tree__min_samples_split': 5}
🎯 Best 5-Fold CV R²   : 83.69%


## ⚔️ STEP 10: ALGORITHM DUEL (DecisionTreeRegressor vs Ridge Regression)
Benchmarking non-linear recursive tree partitioning against regularized linear regression.


In [8]:
# STEP 10: MODEL BENCHMARK
from sklearn.preprocessing import StandardScaler

pipe_ridge = Pipeline([
    ('prep', preprocessor),
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=10.0, random_state=42))
])
pipe_ridge.fit(X_train, y_train)

y_pred_tree = best_model.predict(X_test)
y_pred_ridge = pipe_ridge.predict(X_test)

print("="*65)
print(f"Tuned DecisionTreeRegressor -> Test R²: {r2_score(y_test, y_pred_tree)*100:.2f}% | RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_tree)):.2f} MPa")
print(f"Ridge Regression            -> Test R²: {r2_score(y_test, y_pred_ridge)*100:.2f}% | RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_ridge)):.2f} MPa")
print(f"Tree Improvement Over Linear: +{(r2_score(y_test, y_pred_tree) - r2_score(y_test, y_pred_ridge))*100:.2f}% R² gain!")
print("="*65)


Tuned DecisionTreeRegressor -> Test R²: 80.44% | RMSE: 7.10 MPa
Ridge Regression            -> Test R²: 62.64% | RMSE: 9.81 MPa
Tree Improvement Over Linear: +17.80% R² gain!


## 📊 STEP 11: RESIDUAL ANALYSIS & EVALUATION DIAGNOSTICS
Visualizing residual errors and feature importances.


In [9]:
# STEP 11: EVALUATION REPORT
residuals = y_test - y_pred_tree
mae = mean_absolute_error(y_test, y_pred_tree)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_tree))

print(f"📋 CONCRETE STRENGTH PREDICTION DIAGNOSTICS:")
print(f"   Mean Absolute Error (MAE) : {mae:.2f} MPa")
print(f"   Root Mean Squared Error   : {rmse:.2f} MPa")
print(f"   Coefficient of Det. (R²)  : {r2_score(y_test, y_pred_tree)*100:.2f}%")

# Feature importances
tree_est = best_model.named_steps['tree']
importances = pd.Series(tree_est.feature_importances_, index=X.columns).sort_values(ascending=False)
print(f"\nTop Materials Strength Drivers:\n{importances.round(4)}")


📋 CONCRETE STRENGTH PREDICTION DIAGNOSTICS:
   Mean Absolute Error (MAE) : 4.72 MPa
   Root Mean Squared Error   : 7.10 MPa
   Coefficient of Det. (R²)  : 80.44%

Top Materials Strength Drivers:
cement                0.3662
age                   0.3203
water                 0.1451
blast_furnace_slag    0.0772
fine_aggregate        0.0381
fly_ash               0.0247
superplasticizer      0.0208
coarse_aggregate      0.0077
dtype: float64


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & SMOKE TEST
Deploying the model pipeline and benchmarking sub-millisecond batch mix QA latency.


In [10]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/concrete_strength_tree_pipeline.joblib'
joblib.dump(best_model, model_path)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_mix = pd.DataFrame([{
    'cement': 380.0,
    'blast_furnace_slag': 95.0,
    'fly_ash': 0.0,
    'water': 228.0,
    'superplasticizer': 0.0,
    'coarse_aggregate': 932.0,
    'fine_aggregate': 594.0,
    'age': 28.0
}])

t0 = time.perf_counter()
pred_strength = loaded_pipe.predict(sample_mix)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🏗️ LIVE BATCH MIX QUALITY AUDIT:")
print(f"   Estimated 28-Day Strength: 🧱 {pred_strength:.2f} MPa")
print(f"   Structural Grade         : {'✅ HIGH STRENGTH (>40 MPa)' if pred_strength >= 40 else 'Standard Construction'}")
print(f"   Inference Latency        : {latency_ms:.3f} ms (SLA < 1.0ms: {'PASS' if latency_ms < 1.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/concrete_strength_tree_pipeline.joblib (42,716 bytes)

🏗️ LIVE BATCH MIX QUALITY AUDIT:
   Estimated 28-Day Strength: 🧱 38.24 MPa
   Structural Grade         : Standard Construction
   Inference Latency        : 1.809 ms (SLA < 1.0ms: CHECK)
